<table style="width:100%; border:none; background:transparent;">
  <tr style="border:none;">
    <td style="border:none; text-align:left; vertical-align:middle; width:22%;">
      <img src="./logos/EOSIAL_logo_Sapienza_white_background.bmp" alt="EOSIAL / Sapienza University of Rome" style="max-height:85px;">
    </td>
    <td style="border:none; text-align:center; vertical-align:middle;">
      <h1 style="margin:0; font-size:1.6em;">Flood Mapping Challenge</h1>
      <h3 style="margin:0; font-weight:normal;">Sentinel-1 SAR &mdash; Storm Boris, Nysa (Poland)</h3>
    </td>
    <td style="border:none; text-align:right; vertical-align:middle; width:22%;">
      <img src="./logos/agh_znk_wbr_rgb_150ppi.jpg" alt="AGH University of Krakow" style="max-height:85px;">
    </td>
  </tr>
</table>

**AGH University of Krakow — Aerospace Engineering**  
**Author:** Valerio Pampanoni ([valerio.pampanoni@uniroma1.it](mailto:valerio.pampanoni@uniroma1.it))  

---

## Your Mission

Between 13 and 16 September 2024, [Storm Boris](https://en.wikipedia.org/wiki/2024_Central_European_floods) brought record rainfall to Central Europe, causing severe flooding in Poland's Opole and Lower Silesia voivodeships. The city of **Nysa** was among the worst hit, with over 44,000 people evacuated.

**Your task:** using Sentinel-1 SAR imagery, produce a **binary flood map** showing the areas that were flooded by Storm Boris near Nysa.

You will need to:
1. **Search & download** two Sentinel-1 images from CDSE: one *before* the flood (archive) and one *during/after* the flood (crisis).
2. **Pre-process** both images using ESA SNAP operators.
3. **Detect flooded areas** by comparing the two images.
4. **Export and visualize** your flood map.

### The Rules

Below you will find **all the Python functions** you need to complete this task. Each function is documented with an explanation of what it does and what it returns. **Your job is to figure out in which order to call them and how to connect them** to produce the final flood map.

### What you know about SAR and water

- SAR sensors are **active** (they emit their own microwave pulses), so they work **day and night, through clouds and rain**.
- **Water surfaces** act as specular reflectors: the radar pulse bounces away like a mirror, returning **very low backscatter** to the sensor.
- **Land, vegetation, and buildings** scatter the signal back towards the sensor, producing **higher backscatter**.
- Therefore, **flooded areas appear dark** in SAR images compared to dry land.
- By comparing a pre-flood image with a co-flood image, areas that *became* dark indicate **new flooding**.

---
## 0. Environment Setup

Run the two cells below to load all the libraries you will need.

In [ ]:
import sys
import os
from pathlib import Path
import getpass

import numpy as np
import matplotlib.pyplot as plt
import geopandas as gpd
from shapely.geometry import shape, mapping
from shapely import wkt as shapely_wkt
import rasterio
from rasterio.features import shapes as rio_shapes
from rasterio.transform import from_bounds
import ipywidgets as widgets
from IPython.display import display
import leafmap

# Add the src/ directory so we can import the CDSE helper module
sys.path.insert(0, str(Path.cwd().parent / "src"))
from cdse import get_access_token, search_sentinel1_grd, print_search_results, download_product

print("Base libraries loaded successfully.")

In [ ]:
# Load the SNAP-Python bridge
try:
    from esa_snappy import ProductIO, GPF, HashMap, WKTReader
    import jpy
    print(f"esa_snappy loaded successfully.")
    print(f"SNAP home: {GPF.getDefaultInstance().getOperatorSpiRegistry()}")
except ImportError as e:
    print(f"ERROR: Could not import esa_snappy: {e}")
    print("Make sure ESA SNAP (v12+) is installed and esa_snappy is configured.")
    raise

---
# Part 1 — The Toolkit

Below are all the functions you need. **Read each one carefully** — the docstrings explain what the function does, what inputs it expects, and what it returns. You do **not** need to modify any of these functions.

---
### 1.1 — CDSE Data Access Functions

These functions are already imported from `src/cdse.py`. They allow you to search and download Sentinel-1 data from the Copernicus Data Space Ecosystem (CDSE).

| Function | What it does |
|----------|-------------|
| `get_access_token(username, password)` | Authenticate with CDSE and get an OAuth2 token (valid ~10 min). |
| `search_sentinel1_grd(wkt, start_date, end_date, orbit_direction, product_type)` | Search the CDSE catalogue for Sentinel-1 GRD products that intersect the given WKT geometry and time range. Returns a list of product dictionaries. |
| `print_search_results(products)` | Pretty-print the search results (name, date, size). |
| `download_product(product_id, filename, access_token, output_dir)` | Download a product by its ID. Returns the path to the downloaded `.zip` file. |

**Date format:** `"YYYY-MM-DDTHH:MM:SS.000Z"` (e.g., `"2024-09-01T00:00:00.000Z"`)

**Orbit direction:** `"ASCENDING"` or `"DESCENDING"`. Both images must have the **same orbit direction** so that the incidence angle geometry is consistent and the images are directly comparable.

**Product type:** Use `"IW_GRDH_1S"` for Ground Range Detected, High Resolution, IW mode.

---
### 1.2 — SNAP Processing Functions

These functions wrap ESA SNAP operators to process SAR images. They each take a SNAP product as input and return a new SNAP product.

**What is a SNAP product?** It is a Java object (accessed through `esa_snappy`) that holds raster bands, metadata, and georeferencing information. You create one by reading a Sentinel-1 `.zip` file with `ProductIO.readProduct()`, and each processing function transforms it into a new product.

In [ ]:
def print_product_info(product, label):
    """
    Print basic information about a SNAP product.

    Parameters
    ----------
    product : SNAP Product
        The product to inspect.
    label : str
        A human-readable label (e.g. "Archive", "Crisis").

    Prints the product name, dimensions (pixels), and available band names.
    Useful for verifying each processing step.
    """
    w = product.getSceneRasterWidth()
    h = product.getSceneRasterHeight()
    band_names = list(product.getBandNames())
    print(f"--- {label} ---")
    print(f"  Name       : {product.getName()}")
    print(f"  Dimensions : {w} x {h} pixels")
    print(f"  Bands      : {band_names}")
    print()

In [ ]:
def snap_subset(source_product, wkt_aoi, polarisation="Amplitude_VV"):
    """
    Crop a SNAP product to a geographic area and select a single polarisation band.

    Sentinel-1 IW swaths cover ~250 x 170 km. Your AOI is much smaller, so
    subsetting first dramatically reduces data volume and speeds up all
    subsequent processing.

    Parameters
    ----------
    source_product : SNAP Product
        The full Sentinel-1 product (e.g. from ProductIO.readProduct).
    wkt_aoi : str
        A WKT polygon string defining the area of interest.
    polarisation : str, default "Amplitude_VV"
        The band name to keep. VV polarisation gives the best water/land
        contrast at C-band.

    Returns
    -------
    SNAP Product
        A new product cropped to the AOI with only the selected band.
    """
    params = HashMap()
    params.put("geoRegion", wkt_aoi)
    params.put("bandNames", polarisation)
    params.put("copyMetadata", True)
    return GPF.createProduct("Subset", params, source_product)

In [ ]:
def snap_speckle_filter(source_product, filter_name="Refined Lee", window_size="7x7"):
    """
    Apply an adaptive speckle filter to reduce SAR noise.

    SAR images contain "speckle" — a granular salt-and-pepper noise caused by
    the coherent nature of radar. Speckle is multiplicative (not additive),
    so simple Gaussian smoothing is suboptimal.

    Adaptive filters like Refined Lee estimate local statistics and adjust
    smoothing accordingly:
      - Homogeneous regions (water, flat fields) → strong smoothing
      - Edges and point targets (buildings) → little/no smoothing

    Parameters
    ----------
    source_product : SNAP Product
        Input product (e.g. after subsetting).
    filter_name : str, default "Refined Lee"
        Filter algorithm. Options: "Refined Lee", "Lee Sigma", "Gamma Map",
        "Lee", "Median", "Frost", "Boxcar".
    window_size : str, default "7x7"
        Filter window size. Options: "3x3", "5x5", "7x7", "9x9", etc.

    Returns
    -------
    SNAP Product
        Speckle-filtered product.
    """
    params = HashMap()
    params.put("filter", filter_name)
    wx, wy = window_size.split("x")
    Integer = jpy.get_type("java.lang.Integer")
    params.put("filterSizeX", Integer.parseInt(wx))
    params.put("filterSizeY", Integer.parseInt(wy))
    params.put("estimateENL", True)
    return GPF.createProduct("Speckle-Filter", params, source_product)

In [ ]:
def snap_calibrate(source_product):
    """
    Radiometric calibration: convert raw amplitude to σ⁰ (sigma-nought).

    Raw Sentinel-1 pixel values are uncalibrated digital numbers (DN). To
    compare images acquired at different times or from different sensors, you
    must convert them to a physically meaningful quantity.

    σ⁰ (sigma-nought) is the radar backscatter coefficient: the ratio of
    scattered power per unit area to the incident power density. It is
    returned in **linear scale** (not dB) — you can convert to dB later
    using: dB = 10 * log10(σ⁰).

    Parameters
    ----------
    source_product : SNAP Product
        Input product (should already be speckle-filtered).

    Returns
    -------
    SNAP Product
        Calibrated product with a "Sigma0_VV" band in linear scale.
    """
    params = HashMap()
    params.put("outputSigmaBand", True)
    params.put("outputImageScaleInDb", False)
    return GPF.createProduct("Calibration", params, source_product)

In [ ]:
def snap_terrain_correction(source_product, dem_name="SRTM 3Sec", pixel_spacing_m=20.0):
    """
    Range-Doppler Terrain Correction: orthorectify and geocode to WGS 84.

    SAR images are acquired in "slant range" geometry, which introduces
    geometric distortions (foreshortening, layover, shadow) caused by
    terrain. Terrain Correction uses a DEM to project each pixel onto its
    correct geographic position.

    After this step, the image is in standard WGS 84 geographic coordinates
    and can be overlaid on maps or combined with other geospatial data.

    Note: SNAP automatically downloads the DEM (SRTM) the first time.

    Parameters
    ----------
    source_product : SNAP Product
        Input product (should be calibrated).
    dem_name : str, default "SRTM 3Sec"
        DEM to use for correction.
    pixel_spacing_m : float, default 20.0
        Output pixel spacing in metres.

    Returns
    -------
    SNAP Product
        Terrain-corrected, geocoded product.
    """
    params = HashMap()
    params.put("demName", dem_name)
    params.put("imgResamplingMethod", "BILINEAR_INTERPOLATION")
    params.put("pixelSpacingInMeter", pixel_spacing_m)
    params.put("mapProjection", "WGS84(DD)")
    params.put("saveSelectedSourceBand", True)
    params.put("nodataValueAtSea", True)
    return GPF.createProduct("Terrain-Correction", params, source_product)

---
### 1.3 — I/O and Conversion Functions

These functions help you read data from SNAP products, convert between scales, and export results.

In [ ]:
def snap_band_to_array(product, band_name=None):
    """
    Read a SNAP product band into a NumPy 2D array (float32).

    Parameters
    ----------
    product : SNAP Product
        The product to read from.
    band_name : str or None
        Name of the band to read. If None, the first band is used.

    Returns
    -------
    numpy.ndarray
        2D float32 array of pixel values (shape: height x width).
    """
    if band_name is None:
        band_name = product.getBandNames()[0]
    band = product.getBand(band_name)
    w = band.getRasterWidth()
    h = band.getRasterHeight()
    data = np.zeros(w * h, dtype=np.float32)
    band.readPixels(0, 0, w, h, data)
    return data.reshape(h, w)

In [ ]:
def log_stretch(arr):
    """
    Apply a log10 stretch for displaying linear amplitude / DN values.

    Raw SAR amplitude values have a huge dynamic range. A log10 stretch
    compresses this range so that both dark (water) and bright (land)
    areas are visible in a single plot.

    Parameters
    ----------
    arr : numpy.ndarray
        Linear-scale values.

    Returns
    -------
    numpy.ndarray
        log10-stretched values, with invalid values set to NaN.
    """
    with np.errstate(divide="ignore", invalid="ignore"):
        out = np.log10(arr.astype(np.float32))
    out[~np.isfinite(out)] = np.nan
    return out

In [ ]:
def to_db(linear_array):
    """
    Convert linear backscatter values (σ⁰) to decibels (dB).

    Formula: dB = 10 * log10(linear_value)

    The dB scale compresses the dynamic range and makes the water/land
    contrast much easier to see:
      - Calm water (specular reflection): ~ -20 to -25 dB
      - Bare soil / short vegetation:     ~ -10 to -5 dB
      - Urban / forest:                   ~ -5 to +5 dB

    Parameters
    ----------
    linear_array : numpy.ndarray
        σ⁰ values in linear scale.

    Returns
    -------
    numpy.ndarray
        σ⁰ values in dB. Invalid values (zero, negative) become NaN.
    """
    with np.errstate(divide="ignore", invalid="ignore"):
        db = 10.0 * np.log10(linear_array)
    db[~np.isfinite(db)] = np.nan
    return db

In [ ]:
def normalize(arr, vmin=-25, vmax=0):
    """
    Normalize an array to [0, 1] for RGB display.

    Clips values outside [vmin, vmax] and maps the range linearly to [0, 1].
    The defaults (-25 to 0 dB) span the typical range from calm water to
    bright land in C-band VV.

    Parameters
    ----------
    arr : numpy.ndarray
        Input values (typically dB).
    vmin : float, default -25
        Value mapped to 0.
    vmax : float, default 0
        Value mapped to 1.

    Returns
    -------
    numpy.ndarray
        Array with values in [0, 1].
    """
    out = (arr - vmin) / (vmax - vmin)
    return np.clip(np.nan_to_num(out, nan=0.0), 0, 1)

---
### 1.4 — Useful SNAP and rasterio Operations (not wrapped as functions)

Some operations are simple one-liners. Here is how to use them:

**Reading a Sentinel-1 `.zip` into SNAP:**
```python
product = ProductIO.readProduct("path/to/S1A_IW_GRDH_...zip")
```

**Exporting a SNAP product to GeoTIFF:**
```python
output_path = Path("output/my_image.tif")
ProductIO.writeProduct(product, str(output_path.with_suffix("")), "GeoTIFF")
```
Note: pass the path **without** the `.tif` extension — SNAP adds it automatically.

**Loading a GeoTIFF into NumPy with rasterio:**
```python
with rasterio.open("output/my_image.tif") as src:
    data = src.read(1)              # band 1 as 2D array
    profile = src.profile.copy()    # georeference metadata
    transform = src.transform       # affine transform (pixel ↔ coordinates)
    crs = src.crs                   # coordinate reference system
```

**Writing a raster array to GeoTIFF:**
```python
profile.update(dtype=rasterio.uint8, count=1, compress="lzw", nodata=255)
with rasterio.open("output/my_result.tif", "w", **profile) as dst:
    dst.write(my_array, 1)
```

**Polygonizing a raster mask to vector polygons:**
```python
from rasterio.features import shapes as rio_shapes
from shapely.geometry import shape

polygons = []
for geom, value in rio_shapes(data, mask=(data == 1), connectivity=8, transform=transform):
    if value == 1:
        polygons.append(shape(geom))
```

---
### 1.5 — Visualization Tips

**Displaying a SAR image (dB scale):**
```python
fig, ax = plt.subplots(figsize=(10, 8))
ax.imshow(image_db, cmap="gray", vmin=-25, vmax=0)
ax.set_title("My SAR image [dB]")
ax.axis("off")
plt.show()
```

**False-colour RGB composite** — a classic technique for change detection:
- Assign the **archive** image to the **Red** channel.
- Assign the **crisis** image to the **Green** and **Blue** channels.
- Stack them: `rgb = np.stack([red, green, blue], axis=-1)`
- **Red pixels** = high archive backscatter + low crisis backscatter = **flooded areas!**
- (Normalize both images to [0, 1] before stacking.)

**Interactive map with leafmap:**
```python
m = leafmap.Map(center=[lat, lon], zoom=12)
m.add_raster("path.tif", layer_name="My Layer", colormap="gray", vmin=0, vmax=0.3)
m.add_gdf(my_geodataframe, layer_name="Polygons",
          style={"color": "blue", "fillColor": "cyan", "fillOpacity": 0.5})
m
```

---
# Part 2 — The Challenge

Now it's your turn! Using the toolkit above, build a complete flood mapping workflow.

### What you need to deliver

1. A **binary flood mask** (1 = flooded, 0 = not flooded) saved as a GeoTIFF.
2. **Flood polygons** exported as GeoJSON, with an estimate of the **total flooded area** in hectares.
3. An **interactive map** showing the flood extent on top of the SAR image.

### Key information

| Item | Value |
|------|-------|
| **AOI file** | `../aoi/nysa_aoi.geojson` |
| **Storm dates** | 13–16 September 2024 |
| **Search window** | September 2024 (find a pre-event and a co-event image) |
| **Orbit direction** | Use the same for both images (ASCENDING or DESCENDING) |
| **Product type** | `IW_GRDH_1S` |
| **Polarisation** | VV (better water/land contrast at C-band) |
| **Output directory** | `../output/` |

### Hints (reveal only if stuck!)

<details>
<summary><b>Hint 1:</b> What is the correct processing chain order?</summary>

The standard SNAP processing chain for Sentinel-1 GRD is:

**Read → Subset → Speckle Filter → Calibrate → Terrain Correction → Export**

Apply the same chain to both the archive and the crisis image.
</details>

<details>
<summary><b>Hint 2:</b> How do I detect floods from two SAR images?</summary>

1. Convert both terrain-corrected images to dB.
2. Compute the **difference**: `delta = archive_dB - crisis_dB`.
3. Where `delta` is **large and positive**, backscatter *decreased* → new water.
4. Also check that the crisis image itself shows **low backscatter** (water signature).

A pixel is flooded if:
- `delta_dB > threshold_change` (e.g., 3 dB), **AND**
- `crisis_dB < threshold_water` (e.g., -15 dB)
</details>

<details>
<summary><b>Hint 3:</b> The images have slightly different dimensions!</summary>

After terrain correction, the two images may differ by a few pixels. Crop both to the minimum common size:
```python
min_rows = min(arr1.shape[0], arr2.shape[0])
min_cols = min(arr1.shape[1], arr2.shape[1])
arr1 = arr1[:min_rows, :min_cols]
arr2 = arr2[:min_rows, :min_cols]
```
</details>

<details>
<summary><b>Hint 4:</b> How do I compute flood area from polygons?</summary>

Reproject to a metric CRS (e.g., UTM zone 33N = EPSG:32633) before computing areas:
```python
gdf_metric = gdf.to_crs(epsg=32633)
gdf["area_m2"] = gdf_metric.geometry.area
gdf["area_ha"] = gdf["area_m2"] / 10_000
```
</details>

---
# Part 3 — Your Workspace

Use the cells below to build your solution. Add as many cells as you need.

In [ ]:
# Step 1: Load the AOI
# -----------------------------------------------------------------
# Use gpd.read_file() to load the GeoJSON bounding box.
# Then extract the WKT string of the geometry — you will need it
# as the search footprint when querying CDSE.

# aoi = gpd.read_file("../aoi/nysa_aoi.geojson")
# wkt_aoi = aoi.geometry.iloc[0].wkt
# print(wkt_aoi)

In [ ]:
# Step 2: Search for Sentinel-1 products on CDSE
# -----------------------------------------------------------------
# Authenticate first, then run two searches:
#   - products_pre  : images BEFORE the storm (before 13 Sep 2024)
#   - products_post : images DURING/AFTER the storm (13–20 Sep 2024)
#
# Both searches must use the SAME orbit_direction so the images are
# acquired from the same viewing geometry.
#
# Functions to use:
#   get_access_token(username, password)          → token
#   search_sentinel1_grd(wkt, start, end,
#                        orbit_direction,
#                        product_type)            → list of product dicts
#   print_search_results(products)                → prints a summary table

# username = input("CDSE username: ")
# password = getpass.getpass("CDSE password: ")
# token = get_access_token(username, password)

# products_pre = search_sentinel1_grd(
#     wkt_aoi,
#     start_date="2024-09-01T00:00:00.000Z",
#     end_date  ="2024-09-12T23:59:59.000Z",
#     orbit_direction="ASCENDING",
#     product_type="IW_GRDH_1S",
# )
# products_post = search_sentinel1_grd(
#     wkt_aoi,
#     start_date="2024-09-13T00:00:00.000Z",
#     end_date  ="2024-09-20T23:59:59.000Z",
#     orbit_direction="ASCENDING",
#     product_type="IW_GRDH_1S",
# )
# print("--- Pre-event ---")
# print_search_results(products_pre)
# print("--- Co-event ---")
# print_search_results(products_post)

In [ ]:
# Step 3: Select your archive and crisis images and download them
# -----------------------------------------------------------------
# Inspect the search results and pick one product from each list.
# Use the product "Id" and "Name" fields for the download call.
#
# Function to use:
#   download_product(product_id, filename, access_token, output_dir)
#       → Path to the downloaded .zip file

output_dir = Path("../output")
output_dir.mkdir(exist_ok=True)

# archive_id   = products_pre[0]["Id"]
# archive_name = products_pre[0]["Name"]
# crisis_id    = products_post[0]["Id"]
# crisis_name  = products_post[0]["Name"]

# archive_zip = download_product(archive_id, archive_name, token, output_dir)
# crisis_zip  = download_product(crisis_id,  crisis_name,  token, output_dir)
# print(f"Archive : {archive_zip}")
# print(f"Crisis  : {crisis_zip}")

In [ ]:
# Step 4: Read the products into SNAP
# -----------------------------------------------------------------
# Open each .zip file as a SNAP Product, then inspect it to see
# the available bands and dimensions.
#
# Snippets to use:
#   product = ProductIO.readProduct(str(path_to_zip))
#   print_product_info(product, label)

# archive_raw = ProductIO.readProduct(str(archive_zip))
# crisis_raw  = ProductIO.readProduct(str(crisis_zip))
# print_product_info(archive_raw, "Archive (raw)")
# print_product_info(crisis_raw,  "Crisis  (raw)")

In [ ]:
# Step 5: Pre-process both images — apply the full SNAP chain
# -----------------------------------------------------------------
# Chain: Subset → Speckle Filter → Calibrate → Terrain Correction
# Apply the same four steps to both archive and crisis products.
#
# Functions to use (in order):
#   snap_subset(product, wkt_aoi)               → subsetted product
#   snap_speckle_filter(product)                → speckle-filtered product
#   snap_calibrate(product)                     → calibrated product (σ⁰, linear)
#   snap_terrain_correction(product)            → geocoded product (WGS 84)
#   print_product_info(product, label)          → inspect the result

# --- Archive ---
# archive_sub = snap_subset(archive_raw, wkt_aoi)
# archive_spk = snap_speckle_filter(archive_sub)
# archive_cal = snap_calibrate(archive_spk)
# archive_tc  = snap_terrain_correction(archive_cal)
# print_product_info(archive_tc, "Archive (terrain corrected)")

# --- Crisis ---
# crisis_sub  = snap_subset(crisis_raw, wkt_aoi)
# crisis_spk  = snap_speckle_filter(crisis_sub)
# crisis_cal  = snap_calibrate(crisis_spk)
# crisis_tc   = snap_terrain_correction(crisis_cal)
# print_product_info(crisis_tc, "Crisis  (terrain corrected)")

In [ ]:
# Step 6: Export to GeoTIFF and load into NumPy
# -----------------------------------------------------------------
# Write each terrain-corrected product to disk, then open the .tif
# with rasterio to get a NumPy array plus the georeference metadata
# (profile, transform) you will need later to save the flood mask.
#
# Snippets to use:
#   ProductIO.writeProduct(product, str(path_without_extension), "GeoTIFF")
#
#   with rasterio.open(tif_path) as src:
#       data      = src.read(1)           # 2-D float32 array
#       profile   = src.profile.copy()    # dict with dtype, crs, transform …
#       transform = src.transform         # affine transform pixel ↔ coordinates

# archive_tif = output_dir / "archive_tc.tif"
# crisis_tif  = output_dir / "crisis_tc.tif"
# ProductIO.writeProduct(archive_tc, str(archive_tif.with_suffix("")), "GeoTIFF")
# ProductIO.writeProduct(crisis_tc,  str(crisis_tif.with_suffix("") ), "GeoTIFF")

# with rasterio.open(archive_tif) as src:
#     archive_arr = src.read(1)
#     profile     = src.profile.copy()
#     transform   = src.transform

# with rasterio.open(crisis_tif) as src:
#     crisis_arr = src.read(1)

# Trim to the same shape if they differ by a few pixels after TC
# min_rows = min(archive_arr.shape[0], crisis_arr.shape[0])
# min_cols = min(archive_arr.shape[1], crisis_arr.shape[1])
# archive_arr = archive_arr[:min_rows, :min_cols]
# crisis_arr  = crisis_arr[:min_rows, :min_cols]
# print(f"Array shape: {archive_arr.shape}")

In [ ]:
# Step 7: Convert to dB and visualize both images
# -----------------------------------------------------------------
# σ⁰ in linear scale has a huge dynamic range; dB makes it legible.
# The false-colour composite lets you spot flooding even before
# applying any threshold.
#
# Functions to use:
#   to_db(linear_array)                → dB array (NaN where invalid)
#   normalize(db_array, vmin, vmax)    → float array in [0, 1]
#   np.stack([r, g, b], axis=-1)       → RGB image (H × W × 3)

# archive_db = to_db(archive_arr)
# crisis_db  = to_db(crisis_arr)

# fig, axes = plt.subplots(1, 3, figsize=(18, 6))

# axes[0].imshow(archive_db, cmap="gray", vmin=-25, vmax=0)
# axes[0].set_title("Archive [dB]"); axes[0].axis("off")

# axes[1].imshow(crisis_db, cmap="gray", vmin=-25, vmax=0)
# axes[1].set_title("Crisis [dB]"); axes[1].axis("off")

# False-colour: archive → Red channel, crisis → Green + Blue
# r   = normalize(archive_db)
# g   = normalize(crisis_db)
# b   = normalize(crisis_db)
# rgb = np.stack([r, g, b], axis=-1)
# axes[2].imshow(rgb)
# axes[2].set_title("False-colour composite\n(red = backscatter drop = flooding)")
# axes[2].axis("off")

# plt.tight_layout()
# plt.show()

In [ ]:
# Step 8: Detect flooded areas — change detection
# -----------------------------------------------------------------
# A pixel is classified as flooded when BOTH conditions are true:
#   (1) The backscatter dropped significantly from archive to crisis:
#           archive_db − crisis_db  >  threshold_change
#   (2) The crisis image shows a water-like (low) backscatter:
#           crisis_db               <  threshold_water
#
# Start with the default thresholds below, then adjust them in the
# challenge section to improve your result.

threshold_change = 3.0    # dB — minimum backscatter drop to flag as flooded
threshold_water  = -15.0  # dB — maximum crisis backscatter to confirm water

# flood_mask = (
#     (archive_db - crisis_db > threshold_change) &
#     (crisis_db < threshold_water)
# ).astype(np.uint8)

# n_flooded = int(flood_mask.sum())
# print(f"Flooded pixels : {n_flooded}")
# print(f"Coverage       : {100 * n_flooded / flood_mask.size:.1f} %")

# ---- CHALLENGE -------------------------------------------------------
# Re-run this cell after changing the two thresholds above.
#
# Questions to guide your exploration:
#   - What happens when you LOWER threshold_change (e.g. to 1 dB)?
#     Do you get more flooded area, or more noise?
#   - What happens when you RAISE threshold_water (e.g. to −12 dB)?
#     Can you still capture the flooded fields?
#   - Can you find a combination that matches the ~5 000 ha reported
#     by Copernicus EMS (EMSR756) for the full affected region?
#     (Your AOI is only a part of it, so expect a smaller number.)
# ----------------------------------------------------------------------

In [ ]:
# Step 9: Save the flood mask as GeoTIFF
# -----------------------------------------------------------------
# Update the rasterio profile to match the flood mask (uint8, 1 band),
# then write it to disk. The same transform and CRS you loaded in
# Step 6 will be reused so the output is correctly georeferenced.
#
# Snippet to use:
#   profile.update(dtype=rasterio.uint8, count=1, compress="lzw",
#                  nodata=255, width=..., height=...)
#   with rasterio.open(path, "w", **profile) as dst:
#       dst.write(array, 1)

# flood_tif = output_dir / "flood_mask.tif"
# profile.update(
#     dtype=rasterio.uint8, count=1, compress="lzw", nodata=255,
#     width=flood_mask.shape[1], height=flood_mask.shape[0],
# )
# with rasterio.open(flood_tif, "w", **profile) as dst:
#     dst.write(flood_mask, 1)
# print(f"Flood mask saved → {flood_tif}")

In [ ]:
# Step 10: Polygonize and compute flooded area
# -----------------------------------------------------------------
# Convert the raster mask to vector polygons, build a GeoDataFrame,
# then reproject to UTM zone 33N (EPSG:32633) to compute areas in metres.
#
# Snippets to use:
#   rio_shapes(data, mask=..., connectivity=8, transform=transform)
#       → iterator of (geom_dict, value) pairs
#   shape(geom_dict)                → Shapely geometry
#   gpd.GeoDataFrame(geometry=..., crs="EPSG:4326")
#   gdf.to_crs(epsg=32633)
#   gdf_metric.geometry.area        → area in m²

# polygons = [
#     shape(geom)
#     for geom, val in rio_shapes(
#         flood_mask, mask=flood_mask, connectivity=8, transform=transform
#     )
#     if val == 1
# ]

# gdf = gpd.GeoDataFrame(geometry=polygons, crs="EPSG:4326")
# gdf_metric = gdf.to_crs(epsg=32633)
# gdf["area_ha"] = gdf_metric.geometry.area / 10_000

# total_ha = gdf["area_ha"].sum()
# print(f"Flood polygons  : {len(gdf)}")
# print(f"Total flooded   : {total_ha:.0f} ha")

# gdf.to_file(output_dir / "flood_polygons.geojson", driver="GeoJSON")
# gdf.head()

In [ ]:
# Step 11: Visualize the result on an interactive map
# -----------------------------------------------------------------
# Build a leafmap map centred on Nysa and add:
#   • the crisis SAR image as a background layer
#   • the flood mask as a semi-transparent raster overlay
#   • the flood polygons as a vector layer
#
# Functions / snippets to use:
#   leafmap.Map(center=[lat, lon], zoom=12)
#   m.add_raster(path, layer_name=..., colormap=..., vmin=..., vmax=...)
#   m.add_gdf(gdf, layer_name=..., style={...})

# m = leafmap.Map(center=[50.47, 17.33], zoom=12)

# m.add_raster(str(crisis_tif),  layer_name="Crisis SAR [σ⁰]",
#              colormap="gray", vmin=0, vmax=0.3)
# m.add_raster(str(flood_tif),   layer_name="Flood mask",
#              colormap="Blues", vmin=0, vmax=1, opacity=0.6)
# m.add_gdf(gdf, layer_name="Flood polygons",
#           style={"color": "blue", "fillColor": "cyan",
#                  "fillOpacity": 0.4, "weight": 1})
# m

---
# Part 4 — Validation

Once you have your flood map, consider:

1. **Does the total flooded area seem reasonable?** The [Copernicus EMS activation EMSR756](https://emergency.copernicus.eu/mapping/list-of-components/EMSR756) reported ~5,000 hectares of flooding across the entire affected region. Your AOI covers only part of it, so expect a smaller number.

2. **What happens when you change the thresholds?** Try different values for the change threshold and the water threshold. How sensitive is the result?

3. **Do you see any false positives?** Look for areas flagged as flooded that probably aren't (e.g., terrain shadows, wet fields). What could you do to reduce them?

4. **Try the false-colour RGB composite.** Can you see the flooding clearly before you even apply thresholds?